# Data cleaning

**Purpose:** prepare the data for the Subscriptions targeting analysis. Read the raw files, check and fix their data issues, and save one clean table.

**Inputs** (in `DATA_DIR`)
- `merchants.xlsx` and `merchants.csv`: one row per merchant (industry, first charge date, country, business size). Both are exports of the same table; the csv is used to cross-check the xlsx.
- `payments.xlsx`: one row per merchant per day with payment volume, in cents.

**Steps**
1. Read the files.
2. Check each table on its own: column types, duplicates, empty values. Drop id `0`.
3. Clean merchants (combine xlsx and csv) and payments (parse dates, cents to dollars).
4. Merge on `merchant` and save.

**Output** (`OUTPUT_PATH`, pickle): one row per merchant-day with `date`, `merchant`, the four volume columns in dollars, and the merchant attributes `industry`, `first_charge_date`, `country`, `business_size`.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
OUTPUT_PATH = Path("output/merchant_day.pkl")

## 1. Read data

In [2]:
def read_data(data_dir=DATA_DIR):
    merchants_xlsx = pd.read_excel(data_dir / "merchants.xlsx")
    merchants_csv = pd.read_csv(data_dir / "merchants.csv")
    payments_xlsx = pd.read_excel(data_dir / "payments.xlsx")  # payments has no CSV version
    return merchants_xlsx, merchants_csv, payments_xlsx


merchants_xlsx, merchants_csv, payments_xlsx = read_data()

In [3]:
merchants_xlsx.head()

,merchant,industry,first_charge_date,country,business_size
0,5d03e714,Education,2032-02-13 00:00:00+00:00,US,medium
1,da22f154,Others,2031-10-16 00:00:00+00:00,US,small
2,687eebc8,Software,2032-07-23 00:00:00+00:00,US,small
3,de478470,Software,2033-03-15 00:00:00+00:00,US,small
4,1e719b8a,Business services,2035-02-12 00:00:00+00:00,IT,small


In [4]:
payments_xlsx.head()

,date,merchant,subscription_volume,checkout_volume,payment_link_volume,total_volume
0,2041-05-01T00:00:00Z,5d03e714,0,0,0,425340
1,2041-05-01T00:00:00Z,da22f154,0,0,0,17326
2,2041-05-01T00:00:00Z,687eebc8,79400,0,0,79400
3,2041-05-01T00:00:00Z,de478470,268400,0,0,268400
4,2041-05-01T00:00:00Z,1e719b8a,0,19895,0,19895


## 2. Check each table

### Column types

In [5]:
TABLES = {"merchants (xlsx)": merchants_xlsx, "merchants (csv)": merchants_csv, "payments (xlsx)": payments_xlsx}


def type_summary(df):
    """Count the Python type of every value, per column."""
    return df.apply(lambda col: col.map(lambda x: type(x).__name__).value_counts()).T.fillna(0).astype(int)


for name, table in TABLES.items():
    print(name)
    display(type_summary(table))

merchants (xlsx)


,datetime,int,str
merchant,1,696,22930
industry,0,7,23620
first_charge_date,0,26,23601
country,0,0,23627
business_size,0,0,23627


merchants (csv)


,str
merchant,23627
industry,23627
first_charge_date,23627
country,23627
business_size,23627


payments (xlsx)


,datetime,int,str
date,0,0,1577887
merchant,35,40992,1536860
subscription_volume,0,1577887,0
checkout_volume,0,1577887,0
payment_link_volume,0,1577887,0
total_volume,0,1577887,0


xlsx: some hex ids were parsed as `int` or `datetime`, and `0` is used as a placeholder in `industry` and `first_charge_date`. Dates are text. The csv is all text.

### Duplicates

Key for payments is (merchant, date), since the data is described as merchant-day.

In [6]:
KEYS = {"merchants (xlsx)": ["merchant"], "merchants (csv)": ["merchant"], "payments (xlsx)": ["merchant", "date"]}


def duplicate_summary(df, key):
    # Compare as text: the id column mixes str, int and datetime values.
    as_text = df.astype(str)
    return pd.Series({
        "rows": len(df),
        "duplicated rows": as_text.duplicated().sum(),
        "duplicated keys": as_text.duplicated(key).sum(),
    })


pd.DataFrame({name: duplicate_summary(table, KEYS[name]) for name, table in TABLES.items()}).T

,rows,duplicated rows,duplicated keys
merchants (xlsx),23627,0,7
merchants (csv),23627,0,8
payments (xlsx),1577887,0,22


In [7]:
# Rows whose key is duplicated
for name, table in TABLES.items():
    is_dup = table[KEYS[name]].astype(str).duplicated(keep=False)
    print(name)
    display(table[is_dup].sort_values(KEYS[name], key=lambda col: col.astype(str)).head(10))

merchants (xlsx)


,merchant,industry,first_charge_date,country,business_size
11282,0,Healthcare,2040-12-08 00:00:00+00:00,US,small
11606,0,Merchandise,2038-11-30 00:00:00+00:00,CA,small
13020,0,Education,2041-07-15 00:00:00+00:00,MX,small
13384,0,Others,2041-07-24 00:00:00+00:00,GB,small
18458,0,Business services,2041-12-16 00:00:00+00:00,US,small
20739,0,Others,2042-03-10 00:00:00+00:00,US,small
21151,0,Others,2042-03-26 00:00:00+00:00,GB,small
22686,0,Leisure,2042-05-22 00:00:00+00:00,SG,small


merchants (csv)


,merchant,industry,first_charge_date,country,business_size
11282,0.00E+0,Healthcare,2040-12-08 00:00:00+00:00,US,small
11606,0.00E+0,Merchandise,2038-11-30 00:00:00+00:00,CA,small
13020,0.00E+0,Education,2041-07-15 00:00:00+00:00,MX,small
13384,0.00E+0,Others,2041-07-24 00:00:00+00:00,GB,small
18458,0.00E+0,Business services,2041-12-16 00:00:00+00:00,US,small
20739,0.00E+0,Others,2042-03-10 00:00:00+00:00,US,small
21151,0.00E+0,Others,2042-03-26 00:00:00+00:00,GB,small
22686,0.00E+0,Leisure,2042-05-22 00:00:00+00:00,SG,small
12221,4.72E+10,Clothing & accessory,2037-08-01 00:00:00+00:00,US,small
13304,4.72E+10,Education,2041-01-20 00:00:00+00:00,IT,small


payments (xlsx)


,date,merchant,subscription_volume,checkout_volume,payment_link_volume,total_volume
716215,2041-11-16T00:00:00Z,0,0,0,0,8450
716613,2041-11-16T00:00:00Z,0,0,964,0,964
798331,2041-12-07T00:00:00Z,0,0,0,0,16550
798746,2041-12-07T00:00:00Z,0,0,1426,0,1426
804396,2041-12-09T00:00:00Z,0,0,0,0,27144
806680,2041-12-09T00:00:00Z,0,0,0,0,3696
807087,2041-12-09T00:00:00Z,0,0,954,0,954
821980,2041-12-13T00:00:00Z,0,0,0,0,2041
822367,2041-12-13T00:00:00Z,0,0,1904,0,1904
830366,2041-12-15T00:00:00Z,0,0,0,0,3839


Only id `0` is duplicated: Excel turned ids like `0e12ab34` into 0, merging 8 merchants. In the csv, `4.72E+10` is also shared by 2 merchants, so ids are taken from the xlsx.

### Drop id `0`

It cannot be split back into its 8 merchants.

In [8]:
zero_rows = merchants_xlsx.index[merchants_xlsx["merchant"] == 0]
merchants_xlsx.drop(zero_rows, inplace=True)
merchants_csv.drop(zero_rows, inplace=True)  # same rows as the xlsx
payments_xlsx.drop(payments_xlsx.index[payments_xlsx["merchant"] == 0], inplace=True)

### Empty values

In [9]:
def empty_summary(df):
    text_cols = df.columns[~df.dtypes.map(pd.api.types.is_numeric_dtype)]
    return pd.DataFrame({
        "null": df.isna().sum(),
        "blank string": df.apply(lambda col: col.map(lambda x: isinstance(x, str) and x.strip() == "").sum()),
        "placeholder 0": df[text_cols].apply(lambda col: col.isin([0, "0"]).sum()),
    }).reindex(df.columns).fillna(0).astype(int)


for name, table in TABLES.items():
    print(name)
    display(empty_summary(table))

merchants (xlsx)


,null,blank string,placeholder 0
merchant,0,0,0
industry,0,0,7
first_charge_date,0,0,26
country,0,0,0
business_size,0,0,0


merchants (csv)


,null,blank string,placeholder 0
merchant,0,0,0
industry,0,0,7
first_charge_date,0,0,26
country,0,0,0
business_size,0,0,0


payments (xlsx)


,null,blank string,placeholder 0
date,0,0,0
merchant,0,0,0
subscription_volume,0,0,0
checkout_volume,0,0,0
payment_link_volume,0,0,0
total_volume,0,0,0


No nulls. Missing values are coded as `0`: `industry` (7 merchants) and `first_charge_date` (26).

## 3. Clean

### Merchants

The xlsx and csv have the same rows. Ids come from the xlsx (keeps all digits), other columns from the csv (all text).

In [10]:
SIZE_DTYPE = pd.CategoricalDtype(["small", "medium", "large"], ordered=True)


def build_merchants(merchants_xlsx, merchants_csv):
    other_cols = ["industry", "first_charge_date", "country", "business_size"]
    assert (merchants_xlsx[other_cols].astype(str) == merchants_csv[other_cols]).all().all()

    m = merchants_csv[other_cols].copy()
    m.insert(0, "merchant", merchants_xlsx["merchant"].astype(str))

    m["industry"] = (
        m["industry"]
        .replace("0", pd.NA)
        # "...equipment medium": the trailing "medium" is not the business size
        .str.removesuffix(" medium")
        .astype("category")
    )
    m["first_charge_date"] = pd.to_datetime(m["first_charge_date"].replace("0", pd.NA), utc=True).dt.tz_localize(None)
    m["country"] = m["country"].astype("category")
    m["business_size"] = m["business_size"].astype(SIZE_DTYPE)

    assert m["merchant"].is_unique
    return m.reset_index(drop=True)


merchants = build_merchants(merchants_xlsx, merchants_csv)
merchants.info()

<class 'pandas.DataFrame'>
RangeIndex: 23619 entries, 0 to 23618
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   merchant           23619 non-null  str           
 1   industry           23612 non-null  category      
 2   first_charge_date  23593 non-null  datetime64[us]
 3   country            23619 non-null  category      
 4   business_size      23619 non-null  category      
dtypes: category(3), datetime64[us](1), str(1)
memory usage: 439.1 KB


In [11]:
merchants.head()

,merchant,industry,first_charge_date,country,business_size
0,5d03e714,Education,2032-02-13,US,medium
1,da22f154,Others,2031-10-16,US,small
2,687eebc8,Software,2032-07-23,US,small
3,de478470,Software,2033-03-15,US,small
4,1e719b8a,Business services,2035-02-12,IT,small


### Payments

In [12]:
VOLUME_COLS = ["subscription_volume", "checkout_volume", "payment_link_volume", "total_volume"]


def clean_payments(payments):
    p = payments.copy()
    p["merchant"] = p["merchant"].astype(str)
    p["date"] = pd.to_datetime(p["date"], utc=True).dt.tz_localize(None)
    p[VOLUME_COLS] = p[VOLUME_COLS] / 100  # cents -> dollars
    return p.reset_index(drop=True)

## 4. Merge and save

In [13]:
payments_clean = clean_payments(payments_xlsx)

df = payments_clean.merge(merchants, on="merchant", how="left", validate="many_to_one")

assert df["country"].notna().all(), "some payments have no matching merchant"
assert not df.duplicated(["merchant", "date"]).any()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1577737 entries, 0 to 1577736
Data columns (total 10 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   date                 1577737 non-null  datetime64[us]
 1   merchant             1577737 non-null  str           
 2   subscription_volume  1577737 non-null  float64       
 3   checkout_volume      1577737 non-null  float64       
 4   payment_link_volume  1577737 non-null  float64       
 5   total_volume         1577737 non-null  float64       
 6   industry             1577730 non-null  category      
 7   first_charge_date    1576638 non-null  datetime64[us]
 8   country              1577737 non-null  category      
 9   business_size        1577737 non-null  category      
dtypes: category(3), datetime64[us](2), float64(4), str(1)
memory usage: 88.8 MB


In [14]:
df.head()

,date,merchant,subscription_volume,checkout_volume,payment_link_volume,total_volume,industry,first_charge_date,country,business_size
0,2041-05-01,5d03e714,0.0,0.00,0.0,4253.40,Education,2032-02-13,US,medium
1,2041-05-01,da22f154,0.0,0.00,0.0,173.26,Others,2031-10-16,US,small
2,2041-05-01,687eebc8,794.0,0.00,0.0,794.00,Software,2032-07-23,US,small
3,2041-05-01,de478470,2684.0,0.00,0.0,2684.00,Software,2033-03-15,US,small
4,2041-05-01,1e719b8a,0.0,198.95,0.0,198.95,Business services,2035-02-12,IT,small


In [15]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_pickle(OUTPUT_PATH)  # pickle keeps the dtypes (category, datetime)